# Generate Company Brouchre from company website

In [ ]:
from dotenv import load_dotenv
from openrouter import OpenRouter
from scraper import fetch_website_links, fetch_website_contents
import json
from IPython.display import Markdown, display, update_display
import os

## Loading env variables 
Those are - OPEN_ROUTER_API_KEY

In [13]:
load_dotenv()  # Load environment variables from .env file
MODEL = 'openai/gpt-5-nano'  # OpenRouter model ids are prefixed with the provider
BROCHURE_MODEL = 'openai/gpt-4.1-mini'

open_router_client =  OpenRouter(api_key=os.getenv("OPEN_ROUTER_API_KEY")) 


## Simple python code to scrame all urls from a web page

In [14]:
fetch_website_links('https://sayak2k1maruti.github.io')

['/work',
 '/skill',
 '/contact',
 '/resumeGame/index.html',
 'https://github.com/sayak2k1maruti',
 'https://www.linkedin.com/in/sayak-das-2479901a3',
 'https://www.instagram.com/sayak2k1/',
 'mailto:sayakdas2k1@gmail.com',
 'https://drive.google.com/drive/folders/1pBx5NCZQHfO03B6_KwW0ZucMvpU2-8tG?usp=sharing',
 'mailto:sayakdas2k1@gmail.com',
 '/work',
 '/skill',
 '/contact',
 'mailto:sayakdas2k1@gmail.com',
 'https://github.com/sayak2k1maruti',
 'https://www.linkedin.com/in/sayak-das-2479901a3',
 'https://www.instagram.com/sayak2k1/']

## Now we'll use gpt-5-nano to read the links and repond in a structured JOSN which are relevant 

In [15]:
class LinkScraper:
    _SYSTEM_PROMPT = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
Do not include Terms of Service, Privacy Policy, or any other legal pages
Do not include email links 
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""
    def __init__(self, url):
        self.url = url

    def fetch_links(self):
        return fetch_website_links(self.url)

    def get_user_prompt(self):
        return f"""
Here is the list of links found on the webpage {self.url} - 
Found which of these are relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.

Links (some may be relative links, so you may need to convert them to absolute links):
{self.fetch_links()}
"""
    def get_relevant_links(self):
        try:
            print(f"Scraping contents of the relevant links for {self.url} by calling {MODEL}...")
            response = open_router_client.chat.send(
                model=MODEL,
                messages=[
                    {"role": "system", "content": self._SYSTEM_PROMPT},
                    {"role": "user", "content": self.get_user_prompt()}
                ],
                response_format={"type": "json_object"}
            )
            result =  response.choices[0].message.content
            links = json.loads(result)
            if links.get("links") is None:
                print(f"No relevant links found for {self.url}.")
                return []
            print(f"{len(links['links'])} Relevant links found for {self.url}: {links}")
            return links["links"]
        except Exception as e:
            print(f"Error scraping contents for {self.url}: {e}")
            return []

    def scrape_contents(self):
        relevant_links = self.get_relevant_links()
        result = f' ## Landing Page: \n\n {fetch_website_contents(self.url)} \n\n ## Relevant Links: \n'
        for link in relevant_links:
            url = link["url"]
            print(f"Scraping contents of {url}...")
            content = fetch_website_contents(url)
            result += f' ### {link["type"].title()}: \n\n {content} \n\n'
        return result

In [16]:
Links = LinkScraper('https://www.apple.com')
Links.get_user_prompt()
Links.scrape_contents()

Scraping contents of the relevant links for https://www.apple.com by calling openai/gpt-5-nano...
6 Relevant links found for https://www.apple.com: {'links': [{'type': 'investor relations page', 'url': 'https://investor.apple.com/'}, {'type': 'newsroom', 'url': 'https://www.apple.com/newsroom/'}, {'type': 'leadership page', 'url': 'https://www.apple.com/leadership/'}, {'type': 'careers page', 'url': 'https://www.apple.com/careers/us/'}, {'type': 'business page', 'url': 'https://www.apple.com/business/'}, {'type': 'events page', 'url': 'https://www.apple.com/apple-events/'}]}
Scraping contents of https://investor.apple.com/...
Scraping contents of https://www.apple.com/newsroom/...
Scraping contents of https://www.apple.com/leadership/...
Scraping contents of https://www.apple.com/careers/us/...
Scraping contents of https://www.apple.com/business/...
Scraping contents of https://www.apple.com/apple-events/...


' ## Landing Page: \n\n Apple\n\nApple\nApple\nStore\nMac\niPad\niPhone\nWatch\nVision\nAirPods\nHome\nEntertainment\nAccessories\nSupport\n0\n+\nGifts they’ll love.\nFind their perfect match this holiday.\nShop gifts\niPhone Duo\nHello, hello.\nHello,\nHello.\nhello.\nLearn more\nBuy\nMacBook Pro\nPROLED XDR Display\nLearn more\nBuy\niPhone 18 Pro\nPro further.\nLearn more\nBuy\nApple Watch Series 12\nThe most accurate heart rate sensing in a wearable.\n1\nLearn more\nBuy\niPad\nNow with the speed of the A16 chip and double the starting storage.\nLearn more\nBuy\nAirPods Pro 3\nThe world’s best in-ear Active Noise Cancellation.\nLearn more\nBuy\nApple Upgrade\nLove it. Lease it. Upgrade it.\n2\nLearn more\nApple Card\nGet up to 3% Daily\xa0Cash back with every purchase.\nLearn more\nApply now\nEndless entertainment.\nItem 1\nItem 2\nItem 3\nItem 4\nItem 5\nItem 6\nItem 7\nItem 8\nItem 9\nStream now\nComedy\nNew season.\nStream now\nCrime\nThe only way to save a villain’s missing victi

## Now lets make the broucher

In [19]:
class BrouchreGenerator:
    brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
Also include a short summary of the company and its products/services.
Also add images (which can be rendered in vscode jupyter notebook) if you have the information.
"""
    def __init__(self, company_name, url):
        self.company_name = company_name
        self.url = url
        self.scraper = LinkScraper(url)

    def get_brochure_user_prompt(self):
        user_prompt = f"""
    You are looking at a company called: {self.company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
        user_prompt += self.scraper.scrape_contents()
        user_prompt = user_prompt[:5000] # Truncate if more than 5,000 characters
        return user_prompt

    def create_brochure(self):
        response_stream = open_router_client.chat.send(
            model=BROCHURE_MODEL,
            messages=[
                {"role": "system", "content": self.brochure_system_prompt},
                {"role": "user", "content": self.get_brochure_user_prompt()}
            ],
            response_format={"type": "text"},
            stream = True,
            max_tokens=1000
        )
        res = ''
        display_handler = display(Markdown(res),display_id=True)
        for response in response_stream:
            if response.choices:
                res += response.choices[0].delta.content or ''
            update_display(Markdown(res), display_id=display_handler.display_id)

In [20]:
brochure_generator = BrouchreGenerator("Apple", "https://www.apple.com")
brochure_generator.get_brochure_user_prompt()
brochure_generator.create_brochure()

Scraping contents of the relevant links for https://www.apple.com by calling openai/gpt-5-nano...
4 Relevant links found for https://www.apple.com: {'links': [{'type': 'about page', 'url': 'https://www.apple.com/leadership/'}, {'type': 'about page', 'url': 'https://www.apple.com/newsroom/'}, {'type': 'company page', 'url': 'https://investor.apple.com/'}, {'type': 'careers page', 'url': 'https://www.apple.com/careers/us/'}]}
Scraping contents of https://www.apple.com/leadership/...
Scraping contents of https://www.apple.com/newsroom/...
Scraping contents of https://investor.apple.com/...
Scraping contents of https://www.apple.com/careers/us/...
Scraping contents of the relevant links for https://www.apple.com by calling openai/gpt-5-nano...
5 Relevant links found for https://www.apple.com: {'links': [{'type': 'leadership page', 'url': 'https://www.apple.com/leadership/'}, {'type': 'newsroom page', 'url': 'https://www.apple.com/newsroom/'}, {'type': 'careers page', 'url': 'https://www.ap

# Apple Inc. Brochure

---

## About Apple

Apple is a globally renowned technology company dedicated to designing and manufacturing innovative consumer electronics, software, and services. Known for its seamless integration of hardware and software, Apple continually pushes the boundaries of design and technology to enhance user experiences.

---

## Products & Services

### Flagship Devices

- **iPhone 18 Pro** – Cutting-edge smartphone with advanced capabilities to keep you connected and productive.
- **MacBook Pro** – High-performance laptop equipped with ProLED XDR Display for exceptional visual clarity.
- **Apple Watch Series 12** – Featuring the most accurate heart rate sensing technology in a wearable.
- **iPad** – Now powered by the fast A16 chip with doubled starting storage.
- **AirPods Pro 3** – Industry-leading in-ear Active Noise Cancellation for immersive audio experiences.

### Apple Services

- **Apple Upgrade** – Flexible program to love, lease, and upgrade your device as needed.
- **Apple Card** – Enjoy up to 3% daily cash back on every purchase.
- **Apple TV** – Stream endless entertainment across various genres such as comedy, crime, documentary, drama, thriller, and mystery.
- **Apple Arcade, Music, Fitness+, News+, Podcasts, Books, and Books Store** – Comprehensive ecosystem offering games, music, fitness programs, news, and more.

---

## Entertainment Highlights

- Stream live sporting events like **Formula 1 Grand Prix** and **MLS matches** on Apple TV.
- Watch popular series and programs such as **Ted Lasso**, **Hello Kitty Island Adventure**, and documentaries featuring cultural icons like Awkwafina.
- Enjoy exclusive music interviews and daily hits with Apple Music.

---

## Leadership

Apple is led by a world-class executive team focused on innovation, quality, and global impact:

- **John Ternus** – CEO
- **Eddy Cue** – Senior Vice President, Services and Health
- **Craig Federighi** – Senior Vice President, Software Engineering
- **Greg "Joz" Joswiak** – Senior Vice President, Worldwide Marketing
- **Sabih Khan** – Chief Operating Officer
- **Deirdre O’Brien** – Senior Vice President, Retail and People
- Other key leaders include CFO Kevan Parekh, Chief Hardware Officer Johny Srouji, and Industrial Design VP Molly Anderson.

The board is chaired by Executive Chair Tim Cook and includes distinguished industry leaders ensuring strategic governance.

---

## Company Culture

Apple fosters a culture of innovation, creativity, and inclusion. Employees are empowered to push technological boundaries and create products that profoundly impact the world. The company invests heavily in talent development, diversity, and promoting a collaborative environment where every team member can contribute meaningfully.

---

## Careers at Apple

Apple offers diverse opportunities across hardware design, software engineering, marketing, operations, retail, and communications. Whether you are passionate about cutting-edge technology, creativity in product design, or delivering exceptional customer experiences, Apple provides an inspiring workplace and growth potential.

- Positions available worldwide, including leadership roles and specialized technical careers.
- Strong commitment to employee well-being and professional development.
- Programs such as Apple Camp foster early education and community engagement.

---

## Why Choose Apple?

- **Innovation Leader:** Pioneer in consumer technology with consistent product breakthroughs.
- **Seamless Ecosystem:** Integrated devices and services for a smooth user experience.
- **Customer Focused:** Products designed for usability, accessibility, and performance.
- **Global Reach:** Serving millions of customers worldwide with localized support.
- **Sustainability:** Commitments to environmental responsibility and ethical sourcing.

---

## Contact & Exploring More

Explore more about Apple’s products, services, leadership, and news at:

- [Apple Official Website](https://www.apple.com)
- [Apple Newsroom](https://www.apple.com/newsroom)
- [Apple Careers](https://www.apple.com/careers)

---

## Visual Highlights

![Apple Products](https://www.apple.com/v/home/ac/images/heroes/iphone-18-pro/hero_iphone18pro__f06op46e2s6e_large_2x.jpg)  
*Apple iPhone 18 Pro – Pro further.*

![Apple Watch](https://www.apple.com/v/home/ac/images/heroes/watch-series-12/hero_watch_series12__d6t7eh70b0rm_large_2x.jpg)  
*Apple Watch Series 12 – Most accurate heart rate sensing in wearable technology.*

![MacBook Pro](https://www.apple.com/v/home/ac/images/heroes/macbook-pro-14-and-16/hero_macbookpro_14and16__e2fapz8pfi6e_large_2x.jpg)  
*MacBook Pro with stunning ProLED XDR Display.*

---

Apple continues to transform technology while enriching people's lives globally through innovative products and a vibrant corporate culture. Join Apple or partner with them